**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# In Vivo Maps — Signal-Smoothed Individual Subject Maps
## v1: Smooth-First Strategy

**Motivation:**  
Individual subjects show serious B0/receive-coil inhomogeneity that creates slow spatial gradients in the raw signal. Rather than correcting extracted features post-hoc (v3 approach), we apply a mask-aware Gaussian smooth to the **raw 4D signal** (per echo, per slice, in-plane) before any feature extraction or model inference.

**Methods plotted (3 only):**

| Method | Input | Smoothing applied? |
|--------|-------|--------------------|
| DL-4p  | 40-echo L2-norm | Yes (pre-smooth then L2-norm) |
| Triple (A+B+C) | 43-dim (40-echo + R2*_A,B,C) | Yes (smooth → slope features) |
| DM | Pre-computed .mat files | No (loaded as-is) |

**Key tunable:** `smooth_sigma` in CONFIG — increase if gradient persists, decrease if too blurry.

---

## 1. Imports & GPU

In [ ]:
import os, re, json, time
import numpy as np
import scipy.io as sio
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd
import h5py
from scipy.ndimage import gaussian_filter

import torch
import torch.nn as nn

plt.rcParams.update({
    'font.family'    : 'Arial',
    'font.size'      : 9,
    'axes.labelsize' : 10,
    'axes.titlesize' : 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
})

C_DL_4P  = '#1565C0'
C_TRIPLE = '#6A1B9A'
C_DM     = '#E65100'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Configuration

> **`smooth_sigma`**: Gaussian kernel width in voxels applied in-plane per echo.  
> Start with 2–3 (mild smoothing). Increase toward 5 if gradient/inhomogeneity still visible.  
> Set to 0 to disable smoothing entirely (for comparison).

In [ ]:
CONFIG = {
    # ── Data paths (same as v3) ─────────────────────────────────────────
    'images_mat'       : '../code/images.mat',
    'masks_dir'        : '../GESFIDE_data/GES_ROI',
    'echotimes'        : '../echotimes.mat',
    'dm_results_dir'   : './results/t2snr_results_v5/invivo',
    'model_4p_path'    : './results/t2snr_results_v4/models/t2snr_noisy_4param_v4.pt',
    'model_triple_path': './results/triple_regime_results_v1/models/triple_regime_best.pt',

    # ── Parameter bounds ────────────────────────────────────────────────
    'param_mins'  : np.array([0.0,   0.0025,  1.0e-6,  0.050]),
    'param_maxs'  : np.array([1.0,   0.15,   25.0e-6,  0.200]),
    'R2starA_min' : 2.0,  'R2starA_max': 55.0,
    'R2starB_min' :-30.0, 'R2starB_max': 22.0,
    'R2starC_min' : 2.0,  'R2starC_max': 55.0,

    # ── Sequence ────────────────────────────────────────────────────────
    'n_echoes'    : 40,
    'n_fid'       : 14,
    'n_rephas'    : 16,

    # ── Smoothing ────────────────────────────────────────────────────────
    # Applied to the raw signal in-plane per echo BEFORE any processing.
    # sigma in voxels. Set to 0 to skip.
    'smooth_sigma': 1.0,

    # ── Visualisation ────────────────────────────────────────────────────
    'slice_idx'   : 3,          # default slice; auto-selects best if out of range
    'baseline_cond': 'air',
    'dl_batch_size': 4096,
    'output_dir'  : './results/invivo_smooth_v1',
}

# ── Add these keys to CONFIG (or set separately) ──────────────────
CONFIG['dict_base_path']     = '../subsamples/subsamples_v3'
CONFIG['param_path']         = '../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat'
CONFIG['noisefree_sig_path'] = '../subsamples/subsamples_v3/QuasiRand_t2_200.mat'
CONFIG['dict_key']           = 'Dico40_save'
CONFIG['param_key']          = 'par'
CONFIG['dm_batch_size']      = 200

SE_ECHO = CONFIG['n_fid'] + CONFIG['n_rephas']   # = 30
os.makedirs(CONFIG['output_dir'], exist_ok=True)
FIG_DIR = os.path.join(CONFIG['output_dir'], 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

# ── Methods to run & plot ───────────────────────────────────────────────
METHOD_LIST_NAMES = ['DL-4p', 'Triple (A+B+C)', 'DM']
METHOD_LIST_KEYS  = ['DL_4p', 'Triple',          'DM']
N_METHODS = len(METHOD_LIST_KEYS)

# ── Parameter visualisation spec ────────────────────────────────────────
PARAM_NAMES = ['SO2', 'CBV', 'R',   'T2']
PARAM_SCALE = [100,   100,   1e6,   1000]
PARAM_VIS   = [
    ('SO2 (%)',  0, 100,  (50, 100), 'hot'),
    ('CBV (%)',  1, 100,  (2,   8),  'turbo'),
    ('R (um)',   2, 1e6,  (15,  28),  'pink'),
    ('T2 (ms)',  3, 1000, (40, 120), 'bone'),
]

print(f'SE_ECHO        = {SE_ECHO}')
print(f'smooth_sigma   = {CONFIG["smooth_sigma"]} voxels')
print(f'Methods        = {METHOD_LIST_NAMES}')
print(f'Output dir     : {CONFIG["output_dir"]}')

## 3. Helper functions

In [ ]:
# ── I/O helpers (unchanged from v3) ─────────────────────────────────────

def load_mat(path, key):
    try:
        mat = sio.loadmat(path)
        if key in mat: return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2: data = data.T
            return np.array(data, dtype=np.float32)

def load_images_mat(path):
    try: return sio.loadmat(path)
    except NotImplementedError:
        out = {}
        with h5py.File(path, 'r') as f:
            for k in f.keys():
                if k.startswith('#'): continue
                arr = np.array(f[k][()])
                if arr.ndim >= 2: arr = arr.T
                out[k] = arr
        return out

def key_to_subject_id(img_key):
    m = re.match(r'img_(e\d+)', img_key, flags=re.IGNORECASE)
    return m.group(1).upper() if m else img_key

def key_to_condition(img_key):
    m = re.match(r'img_e\d+(.*)', img_key, flags=re.IGNORECASE)
    raw = m.group(1).lower().strip('_') if m else ''
    for tag, label in [('air','air'),('norm','air'),('hyper','hyper'),('hypo','hypo')]:
        if tag in raw: return label
    return raw or 'unknown'

def find_mask(subj_id):
    d = CONFIG['masks_dir']
    for sfx in ['','_AIR','_HYPER','_HYPO','_air']:
        for ext in ['.nii.gz','.nii']:
            p = os.path.join(d, f'{subj_id}{sfx}_ROI{ext}')
            if os.path.exists(p): return p
    raise FileNotFoundError(f'No mask for {subj_id} in {d}')

def save_fig(fig, name):
    for ext in ['png', 'pdf']:
        p  = os.path.join(FIG_DIR, f'{name}.{ext}')
        kw = {'bbox_inches':'tight', 'facecolor':fig.get_facecolor()}
        if ext == 'png': kw['dpi'] = 300
        fig.savefig(p, **kw)
    print(f'  Saved: {name}')

def load_dm_results(img_key):
    d = CONFIG['dm_results_dir']
    for fname in [f'{img_key}_maps_v5.mat', f'{img_key}_maps.mat',
                  f'{img_key}_maps_r2eff.mat', f'{img_key}_maps_t2cond_v5.mat']:
        p = os.path.join(d, fname)
        if not os.path.exists(p): continue
        for key in ['DM_4param','DM','dm_4param']:
            try:
                arr = load_mat(p, key)
                print(f'    DM: {fname}[{key}] shape={arr.shape}')
                return arr.astype(np.float32)
            except Exception: continue
    print(f'    DM not found for {img_key}')
    return None

def batched_predict(model, x_np, dev=device):
    model.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(x_np), CONFIG['dl_batch_size']):
            xb = torch.tensor(x_np[i:i+CONFIG['dl_batch_size']],
                              dtype=torch.float32).to(dev).contiguous()
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds, axis=0)

def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    return data / np.maximum(np.linalg.norm(data, axis=1, keepdims=True), 1e-12)

def params_inverse(scaled, mins, maxs):
    return (scaled * (maxs - mins) + mins).astype(np.float32)

print('Helpers ready.')

In [ ]:
print('Loading noise-free dictionary...')
dm_sigs_raw = load_mat(CONFIG['noisefree_sig_path'], CONFIG['dict_key'])
dm_pars_raw = load_mat(CONFIG['param_path'],         CONFIG['param_key'])[:, :4]

n_dm = min(len(dm_sigs_raw), len(dm_pars_raw))
dm_sigs_raw, dm_pars_raw = dm_sigs_raw[:n_dm], dm_pars_raw[:n_dm]

# Filter to physiological range and normalise
mask_dm  = np.ones(n_dm, dtype=bool)
for i in range(4):
    mask_dm &= ((dm_pars_raw[:, i] >= CONFIG['param_mins'][i]) &
                (dm_pars_raw[:, i] <= CONFIG['param_maxs'][i]))
valid_dm    = np.all(np.isfinite(dm_sigs_raw), axis=1)
dm_sigs_raw = dm_sigs_raw[mask_dm & valid_dm]
dm_pars_raw = dm_pars_raw[mask_dm & valid_dm]

norms_dm   = np.linalg.norm(np.abs(dm_sigs_raw), axis=1, keepdims=True)
dict_norm  = dm_sigs_raw / np.maximum(norms_dm, 1e-12)
dict_t_gpu = torch.tensor(dict_norm, dtype=torch.float16).to(device)

print(f'Dictionary on GPU: {dict_t_gpu.shape}  '
      f'({dict_t_gpu.element_size()*dict_t_gpu.nelement()/1e6:.0f} MB)')


def gpu_dm_predict(X_norm_smoothed, proc_idx, n_voxels):
    """
    Dot-product DM on the smoothed, L2-normalised signal.
    Returns (n_voxels, 4) physical parameters (not scaled).
    """
    batch_size = CONFIG['dm_batch_size']
    y_pred = np.full((n_voxels, 4), np.nan, dtype=np.float32)
    t0 = time.time(); i = 0
    while i < len(proc_idx):
        sl = proc_idx[i:i + batch_size]
        try:
            batch = torch.tensor(
                X_norm_smoothed[sl], dtype=torch.float16).to(device)
            best  = torch.mm(batch, dict_t_gpu.T).argmax(dim=1).cpu().numpy()
            y_pred[sl] = dm_pars_raw[best]
            del batch; torch.cuda.empty_cache(); i += batch_size
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            batch_size = max(50, batch_size // 2)
            print(f'    OOM — batch_size → {batch_size}')
    print(f'    GPU DM (smoothed): {time.time()-t0:.1f}s  ({len(proc_idx):,} voxels)')
    return y_pred

print('GPU DM ready.')

## 4. Signal smoother

Applies a 2D Gaussian smooth **in-plane** for every echo and every slice.  
Non-brain voxels are filled with the slice's brain mean before smoothing so that the Gaussian does not pull boundary values toward zero, and then zeroed out again after smoothing.

In [ ]:
def smooth_signal_4d(sig4d, mask3d, sigma):
    """
    Gaussian smooth the raw MR signal in-plane (H x W) for every
    echo (T) and slice (S).

    Parameters
    ----------
    sig4d  : (H, W, S, T) float32 — raw signal volume
    mask3d : (H, W, S)    bool/float — brain mask
    sigma  : float — Gaussian kernel width in voxels (0 = no smoothing)

    Returns
    -------
    smoothed : (H, W, S, T) float32
    """
    if sigma <= 0:
        return sig4d.copy()

    H, W, S, T = sig4d.shape
    mask3d = mask3d.astype(bool)
    out = np.zeros_like(sig4d)

    for s in range(S):
        mask_sl = mask3d[:, :, s]          # (H, W)
        if mask_sl.sum() == 0:
            continue
        for t in range(T):
            sl = sig4d[:, :, s, t].copy()  # (H, W)

            # Fill non-brain voxels with the slice brain mean so the
            # Gaussian kernel doesn't blur in artificial zeros at edges
            brain_mean = float(np.nanmean(sl[mask_sl]))
            sl_filled  = sl.copy()
            sl_filled[~mask_sl] = brain_mean

            # Smooth the filled slice
            sl_smooth = gaussian_filter(sl_filled, sigma=sigma)

            # Restore zeros outside the brain
            sl_smooth[~mask_sl] = 0.0

            out[:, :, s, t] = sl_smooth

    return out


# Quick sanity check
_sig   = np.random.rand(16, 16, 3, 40).astype(np.float32)
_mask  = np.ones((16, 16, 3), dtype=bool)
_smth  = smooth_signal_4d(_sig, _mask, sigma=2.0)
assert _smth.shape == _sig.shape, 'Shape mismatch after smoothing'
assert not np.allclose(_sig, _smth), 'Smoothing had no effect'
print(f'smooth_signal_4d OK  max_change={np.abs(_smth - _sig).max():.4f}')

## 5. OLS slope & Triple feature builder

Same formula as v3, but now operates on the **already-smoothed** signal,  
so no additional spatial-normalisation step is needed on the slope features.

In [ ]:
def ols_slope(t_vec, sig_mat):
    """OLS slope of log|S| vs t for each voxel. Returns (N,)."""
    log_s  = np.log(np.maximum(np.abs(sig_mat), 1e-9)).astype(np.float64)
    t      = t_vec.astype(np.float64)
    t_c    = t - t.mean()
    log_sm = log_s - log_s.mean(axis=1, keepdims=True)
    return (log_sm * t_c[None, :]).sum(axis=1) / (t_c ** 2).sum()


def build_triple_input(sig_raw_smooth, sig_norm, T_A, T_B, T_C_rel):
    """
    43-dim input: [40-echo L2-norm | R2*_A | R2*_B | R2*_C]
    Operates on the pre-smoothed signal — no spatial normalisation needed.

    sig_raw_smooth : (N, 40) float32  — smoothed signal, all voxels flattened
    sig_norm       : (N, 40) float32  — L2-normalised smoothed signal
    T_A, T_B, T_C_rel : 1D arrays of echo times for each GESFIDE segment
    """
    n_fid    = CONFIG['n_fid']
    n_rephas = CONFIG['n_rephas']
    r2_lb    = 1.0 / CONFIG['param_maxs'][3]   # lower bound: 1/T2_max

    # Segment the smoothed signal into FID, rephasing, post-SE
    sig_A = sig_raw_smooth[:, :n_fid]
    sig_B = sig_raw_smooth[:, n_fid:n_fid + n_rephas]
    sig_C = sig_raw_smooth[:, n_fid + n_rephas:]

    # OLS slope → R2* estimates for each segment
    R2A = np.maximum(-ols_slope(T_A,     sig_A), r2_lb).astype(np.float32)
    R2B = (-ols_slope(T_B,               sig_B)).astype(np.float32)
    R2C = np.maximum(-ols_slope(T_C_rel, sig_C), r2_lb).astype(np.float32)

    def sc(x, lo, hi):
        return np.clip((x - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)

    fA = sc(R2A, CONFIG['R2starA_min'], CONFIG['R2starA_max'])
    fB = sc(R2B, CONFIG['R2starB_min'], CONFIG['R2starB_max'])
    fC = sc(R2C, CONFIG['R2starC_min'], CONFIG['R2starC_max'])

    return np.concatenate(
        [sig_norm, fA[:, None], fB[:, None], fC[:, None]], axis=1
    ).astype(np.float32)


print('Feature builders ready.')

## 6. Model definitions

In [ ]:
class Clamp01(nn.Module):
    def forward(self, x): return x.clamp(0.0, 1.0)


class Conv1DModel(nn.Module):
    """DL-4p: takes 40-echo L2-norm signal."""
    def __init__(self, n_outputs=4, dropout=0.3, in_dim=40):
        super().__init__()
        self.conv_path = nn.Sequential(
            nn.Conv1d(1,  32, 7, padding=3), nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(64, 128,3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128,256,3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256,256,3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        extra = in_dim - 40
        self.mlp = nn.Sequential(
            nn.Linear(1280+extra, 2048), nn.BatchNorm1d(2048), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(2048, 1024),       nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(1024, 512),        nn.BatchNorm1d(512),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256),        nn.BatchNorm1d(256),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,  n_outputs),
        )
        self.out_act = Clamp01()

    def forward(self, x):
        c = self.conv_path(x[:, :40].unsqueeze(1)).flatten(1)
        return self.out_act(self.mlp(torch.cat([c, x[:, 40:]], dim=1)))


class FiLMLayer(nn.Module):
    def __init__(self, feature_dim, cond_in=1, cond_hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_in, cond_hidden), nn.ReLU(),
            nn.Linear(cond_hidden, 2*feature_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        b = torch.zeros(2*feature_dim); b[:feature_dim] = 1.0
        self.net[-1].bias.data.copy_(b)
        self.feature_dim = feature_dim

    def forward(self, x, cond):
        p = self.net(cond)
        return p[:, :self.feature_dim] * x + p[:, self.feature_dim:]


def _film_backbone(dropout=0.05):
    return nn.Sequential(
        nn.Conv1d(1,  32, 7, padding=3), nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
        nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
        nn.Conv1d(64, 128,3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
        nn.Conv1d(128,256,3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
        nn.Conv1d(256,256,3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
    )


class TripleRegimeModel(nn.Module):
    """Triple (A+B+C): takes 43-dim input [40-echo + R2*_A, R2*_B, R2*_C]."""
    def __init__(self, n_outputs=4, dropout=0.05, film_cond_hidden=32, in_dim=43):
        super().__init__()
        self.conv   = _film_backbone(dropout)
        self.fc1    = nn.Linear(1280, 512); self.bn1 = nn.BatchNorm1d(512)
        self.film1  = FiLMLayer(512, 3, film_cond_hidden)
        self.fc2    = nn.Linear(512,  256); self.bn2 = nn.BatchNorm1d(256)
        self.film2  = FiLMLayer(256, 3, film_cond_hidden)
        self.fc3    = nn.Linear(256,  128); self.bn3 = nn.BatchNorm1d(128)
        self.film3  = FiLMLayer(128, 3, film_cond_hidden)
        self.fc_out = nn.Linear(128, n_outputs)
        self.out_act= Clamp01()
        self.drop   = nn.Dropout(dropout)
        self.relu   = nn.ReLU()

    def forward(self, x):
        c = self.conv(x[:, :40].unsqueeze(1)).flatten(1)
        s = x[:, 40:43]   # R2*_A, R2*_B, R2*_C conditioning
        h = self.drop(self.relu(self.bn1(self.film1(self.fc1(c), s))))
        h = self.drop(self.relu(self.bn2(self.film2(self.fc2(h), s))))
        h = self.drop(self.relu(self.bn3(self.film3(self.fc3(h), s))))
        return self.out_act(self.fc_out(h))


print('Model classes defined.')

## 7. Load models

In [ ]:
def load_model(ModelClass, path, in_dim=40, **kwargs):
    ckpt = torch.load(path, map_location=device)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        state = ckpt['model_state_dict']
    elif isinstance(ckpt, dict) and 'state_dict' in ckpt:
        state = ckpt['state_dict']
    elif isinstance(ckpt, dict) and not any(k in ckpt for k in
             ['model_state_dict','state_dict','param_mins']):
        state = ckpt   # dict of tensors is already the state_dict
    else:
        state = ckpt
    m = ModelClass(in_dim=in_dim, **kwargs).to(device)
    m.load_state_dict(state if isinstance(state, dict) else state, strict=False)
    m.eval()
    return m, ckpt

print('Loading DL-4p ...')
model_4p, ckpt_4p = load_model(Conv1DModel, CONFIG['model_4p_path'], in_dim=40)

print('Loading Triple (A+B+C) ...')
model_triple, ckpt_triple = load_model(TripleRegimeModel, CONFIG['model_triple_path'], in_dim=43)

# Recover param bounds from checkpoints if available
PMINS = np.array(ckpt_4p.get('param_mins', CONFIG['param_mins'])
                 if isinstance(ckpt_4p, dict) else CONFIG['param_mins'])
PMAXS = np.array(ckpt_4p.get('param_maxs', CONFIG['param_maxs'])
                 if isinstance(ckpt_4p, dict) else CONFIG['param_maxs'])

print(f'Models loaded.  PMINS={PMINS}  PMAXS={PMAXS}')

## 8. Load in vivo data & echo times

In [ ]:
mat = load_images_mat(CONFIG['images_mat'])
subj_keys = sorted([k for k in mat if k.lower().startswith('img_')])
print(f'Found {len(subj_keys)} scans:')
for k in subj_keys:
    print(f'  {k}: {np.asarray(mat[k]).shape}  cond={key_to_condition(k)}')

et_mat       = sio.loadmat(CONFIG['echotimes'])
echo_times_s = et_mat['Echotimes'].flatten() / 1000.0

n_fid    = CONFIG['n_fid']
n_rephas = CONFIG['n_rephas']

T_A     = echo_times_s[:n_fid]
T_B     = echo_times_s[n_fid:SE_ECHO]
T_C     = echo_times_s[SE_ECHO:]
T_SE_S  = echo_times_s[SE_ECHO - 1]
T_C_rel = T_C - T_SE_S

print(f'T_SE = {T_SE_S*1e3:.2f} ms')
print(f'Part A echoes: {len(T_A)}  Part B echoes: {len(T_B)}  Part C echoes: {len(T_C)}')

## 9. Process all subjects

Pipeline per scan:
1. Load 4D signal `(H, W, S, T=40)`
2. **Gaussian smooth in-plane per echo** → `sig_smooth`
3. L2-normalise smoothed signal → feed to **DL-4p**
4. Build Triple features from smoothed signal → feed to **Triple**
5. Load pre-computed **DM** `.mat` unchanged

In [ ]:
def process_subject(img_key):
    t_start = time.time()

    # ── Load & orient signal ────────────────────────────────────────────
    sig4d = np.asarray(mat[img_key], dtype=np.float32)
    if sig4d.ndim == 4 and sig4d.shape[0] == 40:
        sig4d = np.transpose(sig4d, (1, 2, 3, 0))   # → (H, W, S, T)
    H, W, S, T = sig4d.shape

    subj_id   = key_to_subject_id(img_key)
    condition = key_to_condition(img_key)

    # ── Brain mask ──────────────────────────────────────────────────────
    mask3d = nib.load(find_mask(subj_id)).get_fdata()
    if mask3d.ndim == 4: mask3d = mask3d[..., 0]
    mask3d = (mask3d > 0).astype(np.float32)

    n_vox = H * W * S

    # ── Step 1: Smooth raw signal ───────────────────────────────────────
    sigma = CONFIG['smooth_sigma']
    t0 = time.time()
    sig_smooth = smooth_signal_4d(sig4d, mask3d, sigma)
    print(f'  {img_key}: {sig4d.shape}  cond={condition}  '
          f'smooth(sigma={sigma}): {time.time()-t0:.1f}s')

    # ── Valid voxel index (from smoothed signal) ────────────────────────
    X_raw_sm = sig_smooth.reshape(-1, T)
    proc_idx = np.where(
        np.isfinite(np.abs(X_raw_sm)).all(axis=1) &
        mask3d.flatten().astype(bool) &
        (np.abs(X_raw_sm).sum(axis=1) > 0)
    )[0]
    print(f'    Valid brain voxels: {proc_idx.size:,}')

    # ── Step 2: L2-normalise smoothed signal ───────────────────────────
    X_norm = euclidean_norm(X_raw_sm)   # (N_vox, 40)

    # ── Step 3: Build Triple features from smoothed signal ─────────────
    t0 = time.time()
    X_tri = build_triple_input(X_raw_sm, X_norm, T_A, T_B, T_C_rel)
    print(f'    Triple feature build: {time.time()-t0:.1f}s  dim={X_tri.shape[1]}')

    # ── Inference helpers ───────────────────────────────────────────────
    def infer(model, X_in, n_out, label):
        out = np.full((n_vox, n_out), np.nan, dtype=np.float32)
        t0  = time.time()
        out[proc_idx] = batched_predict(model, X_in[proc_idx])
        print(f'    {label}: {time.time()-t0:.1f}s')
        return out

    def to_phys(sc):
        p = np.full_like(sc, np.nan)
        p[proc_idx] = params_inverse(sc[proc_idx], PMINS, PMAXS)
        return p.reshape(H, W, S, 4)

    # ── Step 4: Run DL models ───────────────────────────────────────────
    phys_4p  = to_phys(infer(model_4p,     X_norm, 4, 'DL-4p'))
    phys_tri = to_phys(infer(model_triple, X_tri,  4, 'Triple'))

    # ── Step 5: Load pre-computed DM (no re-smoothing) ─────────────────
    dm_raw  = load_dm_results(img_key)
    # phys_dm = (
    #     dm_raw.reshape(H, W, S, 4)
    #     if dm_raw is not None and dm_raw.size == n_vox * 4
    #     else np.full((H, W, S, 4), np.nan, dtype=np.float32)
    # )
    # ── Step 5: DM on smoothed signal (replaces pre-computed load) ─────
    phys_dm = gpu_dm_predict(X_norm, proc_idx, n_vox).reshape(H, W, S, 4)

    result = {
        'subject_id': subj_id,
        'condition' : condition,
        'shape'     : (H, W, S),
        'mask'      : mask3d,
        'DL_4p'     : phys_4p,
        'Triple'    : phys_tri,
        'DM'        : phys_dm,
    }

    # Save parameter maps
    sio.savemat(
        os.path.join(CONFIG['output_dir'], f'{img_key}_maps_smooth.mat'),
        {k: v for k, v in result.items() if isinstance(v, np.ndarray)}
    )
    print(f'    Total: {time.time()-t_start:.1f}s')
    return result


# ── Run all subjects ─────────────────────────────────────────────────────
all_results   = {}
all_roi_stats = []

for img_key in subj_keys:
    print(f'\n{"─"*60}')
    try:
        res = process_subject(img_key)
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'  SKIPPED {img_key}: {e}')
        continue

    all_results[img_key] = res

    # Collect ROI stats
    gm_flat = res['mask'].flatten().astype(bool)
    for mname, mkey in zip(METHOD_LIST_NAMES, METHOD_LIST_KEYS):
        for pi, (pname, punit) in enumerate(zip(PARAM_NAMES, ['%','%','um','ms'])):
            vals = res[mkey][..., pi].flatten()[gm_flat] * PARAM_SCALE[pi]
            vals = vals[np.isfinite(vals)]
            if not len(vals): continue
            all_roi_stats.append({
                'subject'  : res['subject_id'],
                'condition': res['condition'],
                'method'   : mname,
                'parameter': pname,
                'unit'     : punit,
                'mean'     : float(np.nanmean(vals)),
                'std'      : float(np.nanstd(vals)),
                'n_voxels' : len(vals),
            })

df_roi = pd.DataFrame(all_roi_stats)
df_roi.to_csv(os.path.join(CONFIG['output_dir'], 'roi_statistics_smooth.csv'), index=False)
print(f'\nDone. {len(all_results)} scans processed.  ROI rows: {len(df_roi)}.')

## 10. Individual subject parameter maps

One figure per scan: **4 parameters × 3 methods**.  
Slice is taken from `CONFIG['slice_idx']`; if out of range, the slice with the largest brain area is used automatically.

In [ ]:
### ── Multi-slice individual subject figure (styled after reference) ──────────
# Layout: rows = 4 params, cols = (N_SLICES per method) × 3 methods
# Vertical divider lines separate method groups, shared colorbar per row.

SUBJECT_KEY  = 'img_e11air'          # ← change condition if needed
N_SLICES     = 4                      # how many slices to show per method
SLICE_OFFSET = 3                      # skip first N slices (often empty/noisy)

# ── Build the slice list ──────────────────────────────────────────────────────
res     = all_results[SUBJECT_KEY]
H, W, S = res['shape']
mask3d  = res['mask']

# Pick N_SLICES evenly spaced from slices with sufficient brain coverage
brain_coverage = mask3d.sum(axis=(0, 1))
good_slices    = np.where(brain_coverage > brain_coverage.max() * 0.3)[0]
good_slices    = good_slices[SLICE_OFFSET:]                        # drop edge slices
step           = max(len(good_slices) // N_SLICES, 1)
show_slices    = good_slices[::step][:N_SLICES]
print(f'Plotting slices: {show_slices.tolist()}')

# ── Figure geometry ───────────────────────────────────────────────────────────
N_PARAMS  = len(PARAM_VIS)
N_COLS    = N_SLICES * N_METHODS      # total image columns (excl. colorbars)
CB_WIDTH  = 0.12                      # colorbar relative width per param row

fig_w = N_COLS * 1.55 + 0.6          # image columns + colorbar margin
fig_h = N_PARAMS * 2.0

fig, axes = plt.subplots(
    N_PARAMS, N_COLS,
    figsize=(fig_w, fig_h),
    gridspec_kw={'hspace': 0.04, 'wspace': 0.03}
)
fig.patch.set_facecolor('black')
fig.suptitle(
    f"Subject {res['subject_id']}  |  {res['condition']}  "
    f"|  smooth σ={CONFIG['smooth_sigma']}",
    fontsize=10, fontweight='bold', color='white', y=1.01
)

# Method group header positions (column index of group centre)
METHOD_LABELS = {'DM': 'Dictionary Matching', 'DL_4p': 'DL-4p', 'Triple': 'Triple (A+B+C)'}
group_starts  = {mkey: mi * N_SLICES for mi, mkey in enumerate(METHOD_LIST_KEYS)}

# ── Draw images ───────────────────────────────────────────────────────────────
for row, (plabel, pidx, scale, (vmin, vmax), cmap_n) in enumerate(PARAM_VIS):
    cmap_obj = plt.colormaps[cmap_n].copy()
    cmap_obj.set_bad('black')
    last_im = None

    for mi, (mname, mkey) in enumerate(zip(METHOD_LIST_NAMES, METHOD_LIST_KEYS)):
        for si, sl in enumerate(show_slices):
            col     = mi * N_SLICES + si
            ax      = axes[row, col]
            ax.set_facecolor('black')
            mask_sl = mask3d[:, :, sl].astype(bool)

            img = np.ma.array(
                res[mkey][:, :, sl, pidx] * scale,
                mask=~mask_sl
            )
            last_im = ax.imshow(
                np.rot90(img), cmap=cmap_obj,
                vmin=vmin, vmax=vmax, interpolation='nearest'
            )
            ax.axis('off')

            # Slice number label on top of first row
            if row == 0:
                ax.set_title(f'sl {sl}', fontsize=7, color='#888888', pad=2)

    # Shared colorbar on the far right of this row
    cax = fig.add_axes([
        axes[row, -1].get_position().x1 + 0.005,
        axes[row, -1].get_position().y0,
        0.008,
        axes[row, -1].get_position().height
    ])
    cb = fig.colorbar(last_im, cax=cax)
    cb.ax.tick_params(labelsize=6, colors='white')
    cb.outline.set_edgecolor('white')
    cb.set_label(plabel, color='white', fontsize=7, labelpad=4)

    # Row parameter label on the left
    axes[row, 0].text(
        -0.12, 0.5, plabel, ha='right', va='center',
        color='white', fontsize=8,
        transform=axes[row, 0].transAxes, rotation=90
    )

# ── Method group headers ──────────────────────────────────────────────────────
for mi, (mname, mkey) in enumerate(zip(METHOD_LIST_NAMES, METHOD_LIST_KEYS)):
    # Centre of the group in figure coordinates
    col_left  = axes[0, mi * N_SLICES]
    col_right = axes[0, mi * N_SLICES + N_SLICES - 1]
    x_left  = col_left.get_position().x0
    x_right = col_right.get_position().x1
    x_mid   = (x_left + x_right) / 2
    y_top   = axes[0, 0].get_position().y1 + 0.015
    fig.text(x_mid, y_top, mname, ha='center', va='bottom',
             color='white', fontsize=9, fontweight='bold',
             transform=fig.transFigure)

# ── Vertical dividers between method groups ───────────────────────────────────
for mi in range(1, N_METHODS):
    col_prev = axes[0,  mi * N_SLICES - 1]
    col_next = axes[0,  mi * N_SLICES]
    x_div = (col_prev.get_position().x1 + col_next.get_position().x0) / 2
    y0    = axes[-1, 0].get_position().y0
    y1    = axes[ 0, 0].get_position().y1 + 0.02
    line  = plt.Line2D(
        [x_div, x_div], [y0, y1],
        transform=fig.transFigure,
        color='#4488CC', linewidth=1.2, linestyle='-'
    )
    fig.add_artist(line)

save_fig(fig, f'multislice_{SUBJECT_KEY}')
plt.show()
plt.close(fig)

In [ ]:
for img_key, res in all_results.items():
    H, W, S = res['shape']
    mask3d  = res['mask']

    # Auto-select slice
    sl = CONFIG['slice_idx'] if CONFIG['slice_idx'] < S else int(mask3d.sum(axis=(0,1)).argmax())
    mask_sl = mask3d[:, :, sl].astype(bool)

    fig, axes = plt.subplots(
        4, N_METHODS,
        figsize=(N_METHODS * 2.8, 4 * 3.0),
        gridspec_kw={'hspace': 0.05, 'wspace': 0.04}
    )
    fig.patch.set_facecolor('black')
    fig.suptitle(
        f"{res['subject_id']}  |  {res['condition']}  |  slice={sl}  "
        f"|  smooth sigma={CONFIG['smooth_sigma']}",
        fontsize=9, fontweight='bold', color='white', y=1.002
    )

    for row, (plabel, pidx, scale, (vmin, vmax), cmap_n) in enumerate(PARAM_VIS):
        cmap_obj = plt.colormaps[cmap_n].copy()
        cmap_obj.set_bad('black')

        for col, (mname, mkey) in enumerate(zip(METHOD_LIST_NAMES, METHOD_LIST_KEYS)):
            ax = axes[row, col]
            ax.set_facecolor('black')

            img = np.ma.array(
                res[mkey][:, :, sl, pidx] * scale,
                mask=~mask_sl
            )
            im = ax.imshow(
                np.rot90(img), cmap=cmap_obj,
                vmin=vmin, vmax=vmax, interpolation='nearest'
            )
            ax.axis('off')

            # Colorbar on last column only
            if col == N_METHODS - 1:
                cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                cb.ax.tick_params(labelsize=6, colors='white')
                cb.outline.set_edgecolor('white')

            # Column header (method name) on top row
            if row == 0:
                ax.set_title(mname, fontsize=9, color='white', pad=4)

            # Row label (parameter name) on first column
            if col == 0:
                ax.text(
                    -0.06, 0.5, plabel, ha='right', va='center',
                    color='white', fontsize=8,
                    transform=ax.transAxes, rotation=90
                )

    save_fig(fig, f'maps_{img_key}_sl{sl}')
    plt.show()
    plt.close(fig)

In [ ]:
for img_key, res in all_results.items():
    H, W, S = res['shape']
    mask3d  = res['mask']

    # Auto-select slice
    sl = CONFIG['slice_idx'] if CONFIG['slice_idx'] < S else int(mask3d.sum(axis=(0,1)).argmax())
    mask_sl = mask3d[:, :, sl].astype(bool)

    fig, axes = plt.subplots(
        4, N_METHODS,
        figsize=(N_METHODS * 2.8, 4 * 3.0),
        gridspec_kw={'hspace': 0.05, 'wspace': 0.04}
    )
    fig.patch.set_facecolor('black')
    fig.suptitle(
        f"{res['subject_id']}  |  {res['condition']}  |  slice={sl}  "
        f"|  smooth sigma={CONFIG['smooth_sigma']}",
        fontsize=9, fontweight='bold', color='white', y=1.002
    )

    for row, (plabel, pidx, scale, (vmin, vmax), cmap_n) in enumerate(PARAM_VIS):
        cmap_obj = plt.colormaps[cmap_n].copy()
        cmap_obj.set_bad('black')

        for col, (mname, mkey) in enumerate(zip(METHOD_LIST_NAMES, METHOD_LIST_KEYS)):
            ax = axes[row, col]
            ax.set_facecolor('black')

            img = np.ma.array(
                res[mkey][:, :, sl, pidx] * scale,
                mask=~mask_sl
            )
            im = ax.imshow(
                np.rot90(img), cmap=cmap_obj,
                vmin=vmin, vmax=vmax, interpolation='nearest'
            )
            ax.axis('off')

            # Colorbar on last column only
            if col == N_METHODS - 1:
                cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                cb.ax.tick_params(labelsize=6, colors='white')
                cb.outline.set_edgecolor('white')

            # Column header (method name) on top row
            if row == 0:
                ax.set_title(mname, fontsize=9, color='white', pad=4)

            # Row label (parameter name) on first column
            if col == 0:
                ax.text(
                    -0.06, 0.5, plabel, ha='right', va='center',
                    color='white', fontsize=8,
                    transform=ax.transAxes, rotation=90
                )

    save_fig(fig, f'maps_{img_key}_sl{sl}')
    plt.show()
    plt.close(fig)

## 11. ROI bar chart (GM, baseline condition)

Quick sanity check: mean ± SEM across subjects for the baseline (`air`) condition.

In [ ]:
air_df = df_roi[df_roi['condition'] == CONFIG['baseline_cond']]

METHOD_STYLES = [
    ('DL-4p',          C_DL_4P),
    ('Triple (A+B+C)', C_TRIPLE),
    ('DM',             C_DM),
]

fig_roi, axes_roi = plt.subplots(1, 4, figsize=(13, 4),
                                  gridspec_kw={'wspace': 0.40})
fig_roi.suptitle(
    f'GM ROI — {CONFIG["baseline_cond"]}  |  signal-smooth sigma={CONFIG["smooth_sigma"]}',
    fontsize=11, fontweight='bold'
)

for ax, (pname, plabel, punit, pscale) in zip(
        axes_roi,
        zip(PARAM_NAMES,
            ['SO2', 'CBV', 'R', 'T2'],
            ['(%)', '(%)', '(um)', '(ms)'],
            PARAM_SCALE)):
    for xi, (mlabel, color) in enumerate(METHOD_STYLES):
        sub = air_df[(air_df['method'] == mlabel) & (air_df['parameter'] == pname)]
        if not len(sub): continue
        m   = np.nanmean(sub['mean'].values)
        sem = np.nanstd(sub['mean'].values) / max(np.sqrt(len(sub)), 1)
        ax.bar(xi, m, 0.7, color=color, edgecolor='black', lw=0.5)
        ax.errorbar(xi, m, yerr=sem, fmt='none', color='black', capsize=4, lw=1.2)
        ax.text(xi, m + sem * 1.5, f'{m:.2f}',
                ha='center', va='bottom', fontsize=6.5)

    ax.set_xticks(range(len(METHOD_STYLES)))
    ax.set_xticklabels([s[0] for s in METHOD_STYLES], rotation=20, ha='right', fontsize=8)
    ax.set_ylabel(f'{plabel} {punit}', fontsize=9)
    ax.set_title(plabel, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.4)
    ax.set_axisbelow(True)

fig_roi.tight_layout()
save_fig(fig_roi, f'roi_gm_barplot_smooth_sigma{CONFIG["smooth_sigma"]}')
plt.show()

# Summary table
print(f"\n{'Method':<22}" + ''.join(f"  {p+' '+u:>16}"
      for p, u in zip(PARAM_NAMES, ['%','%','um','ms'])))
print('-' * 90)
for mlabel, _ in METHOD_STYLES:
    row = f'{mlabel:<22}'
    for pname in PARAM_NAMES:
        sub = air_df[(air_df['method'] == mlabel) & (air_df['parameter'] == pname)]
        if len(sub):
            row += f'  {np.nanmean(sub["mean"].values):>8.2f} +/- {np.nanstd(sub["mean"].values):>5.2f}'
        else:
            row += f'  {"N/A":>16}'
    print(row)